In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

# --- PREPARATION 1: Centered FFT functions ---
def fft2c(im):
    """Centered 2D FFT"""
    return np.fft.fftshift(np.fft.fft2(np.fft.ifftshift(im)))

def ifft2c(d):
    """Centered 2D inverse FFT"""
    return np.fft.fftshift(np.fft.ifft2(np.fft.ifftshift(d)))

def fftc(im):
    """Centered 1D FFT"""
    return np.fft.fftshift(np.fft.fft(np.fft.ifftshift(im)))

def ifftc(d):
    """Centered 1D inverse FFT"""
    return np.fft.fftshift(np.fft.ifft(np.fft.ifftshift(d)))

# --- PREPARATION 2 & 3: Display helper functions ---
def imshow_gray(im, title="", vrange=None):
    """Display a grayscale image"""
    plt.figure()
    if vrange is not None:
        plt.imshow(im, cmap='gray', vmin=vrange[0], vmax=vrange[1])
    else:
        plt.imshow(im, cmap='gray')
    plt.title(title)
    plt.colorbar()
    plt.axis('off')
    plt.show()

def imshow_kspace(d, title=""):
    """Display k-space magnitude spectrum using log scale"""
    plt.figure()
    plt.imshow(np.log(np.abs(d) + 1), cmap='gray')
    plt.title(title)
    plt.colorbar()
    plt.axis('off')
    plt.show()

# --- PREPARATION 5: Load data ---
data = loadmat('ge_phantom.mat')
# Inspect what variables are inside:
print(data.keys())

In [ ]:
img = plt.imread('test_image.jpeg')
m_ref = np.mean(img, axis=2).astype(np.float64)  # Convert to grayscale
Nx, Ny = m_ref.shape
print(f"Image size: {Nx} x {Ny}")

In [ ]:
import os
os.makedirs('figures', exist_ok=True)

imshow_gray(m_ref, "1.1: Reference Image")

# Save figure
plt.figure(); plt.imshow(m_ref, cmap='gray'); plt.title('Reference Image'); plt.axis('off')
plt.savefig('figures/p11_ref.png', bbox_inches='tight', dpi=150); plt.close()

In [ ]:
M_ref = fft2c(m_ref)
imshow_kspace(M_ref, "1.1: Magnitude Spectrum of m_ref")

# Save figure
plt.figure(); plt.imshow(np.log(np.abs(M_ref)+1), cmap='gray'); plt.title('Magnitude Spectrum'); plt.axis('off')
plt.savefig('figures/p11_spectrum.png', bbox_inches='tight', dpi=150); plt.close()

In [ ]:
# === PART 1.2 ===

m_down2 = m_ref[::2, ::2]
print(f"Downsampled image size: {m_down2.shape}")

imshow_gray(m_down2, "1.2: Downsampled by 2")
imshow_kspace(fft2c(m_down2), "1.2: Magnitude Spectrum (downsampled by 2)")

#figure saving 
plt.figure(); plt.imshow(m_down2, cmap='gray'); plt.title('Downsampled by 2'); plt.axis('off')
plt.savefig('figures/p12_down2.png', bbox_inches='tight', dpi=150); plt.close()

plt.figure(); plt.imshow(np.log(np.abs(fft2c(m_down2))+1), cmap='gray'); plt.title('Spectrum (downsampled by 2)'); plt.axis('off')
plt.savefig('figures/p12_spectrum.png', bbox_inches='tight', dpi=150); plt.close()

In [ ]:
# === PART 1.3 ===

# Spectrum of downsampled image
M_down2 = fft2c(m_down2)

# Zero-pad spectrum to Nx x Ny
M_zp = np.zeros((Nx, Ny), dtype=complex)
sx, sy = m_down2.shape
start_x = (Nx - sx) // 2
start_y = (Ny - sy) // 2
M_zp[start_x:start_x+sx, start_y:start_y+sy] = M_down2

# Display zero-padded spectrum
imshow_kspace(M_zp, "1.3: Zero-Padded Magnitude Spectrum")

# Reconstruct image
m_zp = np.abs(ifft2c(M_zp))
imshow_gray(m_zp, "1.3: Zero-Padded Image")

# Normalize each image by its own max
m_ref_norm = m_ref / np.max(m_ref)
m_zp_norm = m_zp / np.max(m_zp)

# Error image
imshow_gray(np.abs(m_ref_norm - m_zp_norm), "1.3: Error Image")

# PSNR and SSIM
print(f"PSNR: {psnr(m_ref_norm, m_zp_norm, data_range=1.0):.2f} dB")
print(f"SSIM: {ssim(m_ref_norm, m_zp_norm, data_range=1.0):.4f}")

# Save figures
plt.figure(); plt.imshow(np.log(np.abs(M_zp)+1), cmap='gray'); plt.title('Zero-Padded Spectrum (2x)'); plt.axis('off')
plt.savefig('figures/p13_zp_spectrum.png', bbox_inches='tight', dpi=150); plt.close()

plt.figure(); plt.imshow(m_zp, cmap='gray'); plt.title('Zero-Padded Image (2x)'); plt.axis('off')
plt.savefig('figures/p13_zp_image.png', bbox_inches='tight', dpi=150); plt.close()

plt.figure(); plt.imshow(np.abs(m_ref_norm - m_zp_norm), cmap='gray'); plt.title('Error Image (2x)'); plt.axis('off')
plt.savefig('figures/p13_error.png', bbox_inches='tight', dpi=150); plt.close()

In [ ]:
# === PART 1.4 ===

# Downsample by 4
m_down4 = m_ref[::4, ::4]
print(f"Downsampled (by 4) image size: {m_down4.shape}")

imshow_gray(m_down4, "1.4: Downsampled by 4")

# FFT and zero-pad to Nx x Ny
M_down4 = fft2c(m_down4)
M_zp4 = np.zeros((Nx, Ny), dtype=complex)
sx4, sy4 = m_down4.shape
start_x4 = (Nx - sx4) // 2
start_y4 = (Ny - sy4) // 2
M_zp4[start_x4:start_x4+sx4, start_y4:start_y4+sy4] = M_down4

imshow_kspace(M_zp4, "1.4: Zero-Padded Magnitude Spectrum (4x)")

# Reconstruct image
m_zp4 = np.abs(ifft2c(M_zp4))
imshow_gray(m_zp4, "1.4: Zero-Padded Image (4x)")

# Error image
m_zp4_norm = m_zp4 / np.max(m_zp4)
imshow_gray(np.abs(m_ref_norm - m_zp4_norm), "1.4: Error Image (4x)")

# PSNR and SSIM
print(f"PSNR: {psnr(m_ref_norm, m_zp4_norm, data_range=1.0):.2f} dB")
print(f"SSIM: {ssim(m_ref_norm, m_zp4_norm, data_range=1.0):.4f}")

# Save figures
plt.figure(); plt.imshow(m_down4, cmap='gray'); plt.title('Downsampled by 4'); plt.axis('off')
plt.savefig('figures/p14_down4.png', bbox_inches='tight', dpi=150); plt.close()

plt.figure(); plt.imshow(np.log(np.abs(M_zp4)+1), cmap='gray'); plt.title('Zero-Padded Spectrum (4x)'); plt.axis('off')
plt.savefig('figures/p14_zp_spectrum.png', bbox_inches='tight', dpi=150); plt.close()

plt.figure(); plt.imshow(m_zp4, cmap='gray'); plt.title('Zero-Padded Image (4x)'); plt.axis('off')
plt.savefig('figures/p14_zp_image.png', bbox_inches='tight', dpi=150); plt.close()

plt.figure(); plt.imshow(np.abs(m_ref_norm - m_zp4_norm), cmap='gray'); plt.title('Error Image (4x)'); plt.axis('off')
plt.savefig('figures/p14_error.png', bbox_inches='tight', dpi=150); plt.close()

In [ ]:
# === PART 2.1: k-space Data ===

# Extract k-space data from the .mat file
kdata = data['kdata']
Nro, Npe = kdata.shape
print(f"k-space data size: Nro={Nro}, Npe={Npe}, dtype={kdata.dtype}")

# Display magnitude spectrum
imshow_kspace(kdata, "2.1: k-space Magnitude Spectrum")

# Compute complex-valued MRI image
m_f = ifft2c(kdata)

# Display magnitude, phase, real, imaginary
imshow_gray(np.abs(m_f), "2.1: Magnitude Image")
imshow_gray(np.angle(m_f), "2.1: Phase Image")
imshow_gray(np.real(m_f), "2.1: Real Part")
imshow_gray(np.imag(m_f), "2.1: Imaginary Part")

# Reference image for the rest of the homework
m_ref_mri = np.abs(m_f)
max_val = np.max(m_ref_mri)
print(f"max_val = {max_val:.4f}")

# Save figures
plt.figure(); plt.imshow(np.log(np.abs(kdata)+1), cmap='gray'); plt.title('k-space Magnitude Spectrum'); plt.axis('off')
plt.savefig('figures/p21_kspace.png', bbox_inches='tight', dpi=150); plt.close()

plt.figure(); plt.imshow(np.abs(m_f), cmap='gray'); plt.title('Magnitude Image'); plt.axis('off')
plt.savefig('figures/p21_magnitude.png', bbox_inches='tight', dpi=150); plt.close()

plt.figure(); plt.imshow(np.angle(m_f), cmap='gray'); plt.title('Phase Image'); plt.axis('off')
plt.savefig('figures/p21_phase.png', bbox_inches='tight', dpi=150); plt.close()

plt.figure(); plt.imshow(np.real(m_f), cmap='gray'); plt.title('Real Part'); plt.axis('off')
plt.savefig('figures/p21_real.png', bbox_inches='tight', dpi=150); plt.close()

plt.figure(); plt.imshow(np.imag(m_f), cmap='gray'); plt.title('Imaginary Part'); plt.axis('off')
plt.savefig('figures/p21_imag.png', bbox_inches='tight', dpi=150); plt.close()

In [ ]:
# === PART 2.2: Skipping lines in k-space ===

# Downsample k-space by 2 in PE direction (remove every other line)
kdata_skip2 = kdata[:, ::2]
print(f"Downsampled k-space size: {kdata_skip2.shape}")

# Display magnitude spectrum and image
imshow_kspace(kdata_skip2, "2.2: k-space (every other PE line kept)")
m_skip2 = np.abs(ifft2c(kdata_skip2))
imshow_gray(m_skip2, "2.2: Magnitude Image (skip 2 in PE)")

# Save figures
plt.figure(); plt.imshow(np.log(np.abs(kdata_skip2)+1), cmap='gray'); plt.title('k-space (skip 2 PE)'); plt.axis('off')
plt.savefig('figures/p22_kspace.png', bbox_inches='tight', dpi=150); plt.close()

plt.figure(); plt.imshow(m_skip2, cmap='gray'); plt.title('Magnitude Image (skip 2 PE)'); plt.axis('off')
plt.savefig('figures/p22_image.png', bbox_inches='tight', dpi=150); plt.close()

In [ ]:
# === PART 2.3: Zero-fill every other PE line (downsample by 2) ===

# Set every other PE line to zero (keep same array size)
kdata_zf2 = kdata.copy()
kdata_zf2[:, 1::2] = 0  # zero out every other PE line

imshow_kspace(kdata_zf2, "2.3: k-space (every other PE line zeroed)")

m_zf2 = np.abs(ifft2c(kdata_zf2))
imshow_gray(m_zf2, "2.3: Magnitude Image (zero-filled, skip 2 PE)")

# Error image with [0, 0.2*max_val] window
err_23 = np.abs(m_zf2 - m_ref_mri)
imshow_gray(err_23, "2.3: Error Image", vrange=[0, 0.2*max_val])

# PSNR and SSIM (normalize by max_val)
print(f"PSNR: {psnr(m_ref_mri/max_val, m_zf2/max_val, data_range=1.0):.2f} dB")
print(f"SSIM: {ssim(m_ref_mri/max_val, m_zf2/max_val, data_range=1.0):.4f}")

# Save figures
plt.figure(); plt.imshow(np.log(np.abs(kdata_zf2)+1), cmap='gray'); plt.title('k-space (zero-filled skip 2 PE)'); plt.axis('off')
plt.savefig('figures/p23_kspace.png', bbox_inches='tight', dpi=150); plt.close()

plt.figure(); plt.imshow(m_zf2, cmap='gray'); plt.title('Magnitude Image (zero-filled skip 2 PE)'); plt.axis('off')
plt.savefig('figures/p23_image.png', bbox_inches='tight', dpi=150); plt.close()

plt.figure(); plt.imshow(err_23, cmap='gray', vmin=0, vmax=0.2*max_val); plt.title('Error Image'); plt.axis('off')
plt.savefig('figures/p23_error.png', bbox_inches='tight', dpi=150); plt.close()

In [ ]:
# === PART 2.4: Zero-fill every other PE line (downsample by 4) ===

# Set 3 out of every 4 PE lines to zero
kdata_zf4 = kdata.copy()
kdata_zf4[:, 1::4] = 0
kdata_zf4[:, 2::4] = 0
kdata_zf4[:, 3::4] = 0

imshow_kspace(kdata_zf4, "2.4: k-space (downsample by 4 in PE, zero-filled)")

m_zf4 = np.abs(ifft2c(kdata_zf4))
imshow_gray(m_zf4, "2.4: Magnitude Image (zero-filled, skip 4 PE)")

# Error image with [0, 0.2*max_val] window
err_24 = np.abs(m_zf4 - m_ref_mri)
imshow_gray(err_24, "2.4: Error Image", vrange=[0, 0.2*max_val])

# PSNR and SSIM (normalize by max_val)
print(f"PSNR: {psnr(m_ref_mri/max_val, m_zf4/max_val, data_range=1.0):.2f} dB")
print(f"SSIM: {ssim(m_ref_mri/max_val, m_zf4/max_val, data_range=1.0):.4f}")

# Save figures
plt.figure(); plt.imshow(np.log(np.abs(kdata_zf4)+1), cmap='gray'); plt.title('k-space (zero-filled skip 4 PE)'); plt.axis('off')
plt.savefig('figures/p24_kspace.png', bbox_inches='tight', dpi=150); plt.close()

plt.figure(); plt.imshow(m_zf4, cmap='gray'); plt.title('Magnitude Image (zero-filled skip 4 PE)'); plt.axis('off')
plt.savefig('figures/p24_image.png', bbox_inches='tight', dpi=150); plt.close()

plt.figure(); plt.imshow(err_24, cmap='gray', vmin=0, vmax=0.2*max_val); plt.title('Error Image'); plt.axis('off')
plt.savefig('figures/p24_error.png', bbox_inches='tight', dpi=150); plt.close()

In [ ]:
# === PART 2.5: Acquiring fewer lines — central half of PE ===

# Keep only central 128 PE lines (remove first 64 and last 64)
kdata_central_half = kdata[:, Npe//4 : 3*Npe//4]
print(f"Central half k-space size: {kdata_central_half.shape}")

imshow_kspace(kdata_central_half, "2.5: k-space (central half PE)")
m_central_half = np.abs(ifft2c(kdata_central_half))
imshow_gray(m_central_half, "2.5: Magnitude Image (central half PE)")

# Save figures
plt.figure(); plt.imshow(np.log(np.abs(kdata_central_half)+1), cmap='gray'); plt.title('k-space (central half PE)'); plt.axis('off')
plt.savefig('figures/p25_kspace.png', bbox_inches='tight', dpi=150); plt.close()

plt.figure(); plt.imshow(m_central_half, cmap='gray'); plt.title('Magnitude Image (central half PE)'); plt.axis('off')
plt.savefig('figures/p25_image.png', bbox_inches='tight', dpi=150); plt.close()

In [ ]:
# === PART 2.6: Central half of PE, zero-fill the rest ===

# Keep central half, zero-fill outer lines
kdata_26 = np.zeros_like(kdata)
kdata_26[:, Npe//4 : 3*Npe//4] = kdata[:, Npe//4 : 3*Npe//4]

imshow_kspace(kdata_26, "2.6: k-space (central half PE, zero-filled)")

m_26 = np.abs(ifft2c(kdata_26))
imshow_gray(m_26, "2.6: Magnitude Image (central half PE, zero-filled)")

# Error image
err_26 = np.abs(m_26 - m_ref_mri)
imshow_gray(err_26, "2.6: Error Image", vrange=[0, 0.2*max_val])

# PSNR and SSIM
print(f"PSNR: {psnr(m_ref_mri/max_val, m_26/max_val, data_range=1.0):.2f} dB")
print(f"SSIM: {ssim(m_ref_mri/max_val, m_26/max_val, data_range=1.0):.4f}")

# Save figures
plt.figure(); plt.imshow(np.log(np.abs(kdata_26)+1), cmap='gray'); plt.title('k-space (central half PE, zero-filled)'); plt.axis('off')
plt.savefig('figures/p26_kspace.png', bbox_inches='tight', dpi=150); plt.close()

plt.figure(); plt.imshow(m_26, cmap='gray'); plt.title('Magnitude Image (central half PE, zero-filled)'); plt.axis('off')
plt.savefig('figures/p26_image.png', bbox_inches='tight', dpi=150); plt.close()

plt.figure(); plt.imshow(err_26, cmap='gray', vmin=0, vmax=0.2*max_val); plt.title('Error Image'); plt.axis('off')
plt.savefig('figures/p26_error.png', bbox_inches='tight', dpi=150); plt.close()

In [ ]:
# === PART 2.7: Central 1/4th of PE, zero-fill the rest ===

# Keep central 64 PE lines (1/4th), zero-fill outer lines
kdata_27 = np.zeros_like(kdata)
kdata_27[:, 3*Npe//8 : 5*Npe//8] = kdata[:, 3*Npe//8 : 5*Npe//8]

imshow_kspace(kdata_27, "2.7: k-space (central 1/4 PE, zero-filled)")

m_27 = np.abs(ifft2c(kdata_27))
imshow_gray(m_27, "2.7: Magnitude Image (central 1/4 PE, zero-filled)")

# Error image
err_27 = np.abs(m_27 - m_ref_mri)
imshow_gray(err_27, "2.7: Error Image", vrange=[0, 0.2*max_val])

# PSNR and SSIM
print(f"PSNR: {psnr(m_ref_mri/max_val, m_27/max_val, data_range=1.0):.2f} dB")
print(f"SSIM: {ssim(m_ref_mri/max_val, m_27/max_val, data_range=1.0):.4f}")

# Save figures
plt.figure(); plt.imshow(np.log(np.abs(kdata_27)+1), cmap='gray'); plt.title('k-space (central 1/4 PE, zero-filled)'); plt.axis('off')
plt.savefig('figures/p27_kspace.png', bbox_inches='tight', dpi=150); plt.close()

plt.figure(); plt.imshow(m_27, cmap='gray'); plt.title('Magnitude Image (central 1/4 PE, zero-filled)'); plt.axis('off')
plt.savefig('figures/p27_image.png', bbox_inches='tight', dpi=150); plt.close()

plt.figure(); plt.imshow(err_27, cmap='gray', vmin=0, vmax=0.2*max_val); plt.title('Error Image'); plt.axis('off')
plt.savefig('figures/p27_error.png', bbox_inches='tight', dpi=150); plt.close()

In [ ]:
# === PART 3.1: Phase-compensated image (central ±1/8, 25%, 64 PE lines) ===

# Low-resolution k-space: central ±Npe/8 = ±32 lines → 64 total
half_w = Npe // 8  # 32
center = Npe // 2  # 128
kdata_low31 = np.zeros_like(kdata)
kdata_low31[:, center - half_w : center + half_w] = kdata[:, center - half_w : center + half_w]

# Low-resolution complex image
m_low31 = ifft2c(kdata_low31)

# Phase image p(x,y) = exp(j * angle(m_low))
p31 = np.exp(1j * np.angle(m_low31))
imshow_gray(np.angle(m_low31), "3.1: Phase Image (central 25%)")

# Phase-compensated full k-space image
m_pc31 = np.maximum(np.real(m_f * np.conj(p31)), 0)
imshow_gray(m_pc31, "3.1: Phase-Compensated Image")

# Error image
err_31 = np.abs(m_pc31 - m_ref_mri)
imshow_gray(err_31, "3.1: Error Image", vrange=[0, 0.2 * max_val])

# PSNR and SSIM
print(f"PSNR: {psnr(m_ref_mri / max_val, m_pc31 / max_val, data_range=1.0):.2f} dB")
print(f"SSIM: {ssim(m_ref_mri / max_val, m_pc31 / max_val, data_range=1.0):.4f}")

# Save figures
plt.figure(); plt.imshow(np.angle(m_low31), cmap='gray'); plt.title('Phase Image (central 25%)'); plt.colorbar(); plt.axis('off')
plt.savefig('figures/p31_phase.png', bbox_inches='tight', dpi=150); plt.close()

plt.figure(); plt.imshow(m_pc31, cmap='gray'); plt.title('Phase-Compensated Image (25%)'); plt.axis('off')
plt.savefig('figures/p31_pc.png', bbox_inches='tight', dpi=150); plt.close()

plt.figure(); plt.imshow(err_31, cmap='gray', vmin=0, vmax=0.2 * max_val); plt.title('Error Image (25%)'); plt.axis('off')
plt.savefig('figures/p31_error.png', bbox_inches='tight', dpi=150); plt.close()

In [ ]:
# === PART 3.2: Phase-compensated image (central ±1/16, 12.5%, 32 PE lines) ===

half_w = Npe // 16  # 16
kdata_low32 = np.zeros_like(kdata)
kdata_low32[:, center - half_w : center + half_w] = kdata[:, center - half_w : center + half_w]

m_low32 = ifft2c(kdata_low32)
p32 = np.exp(1j * np.angle(m_low32))
imshow_gray(np.angle(m_low32), "3.2: Phase Image (central 12.5%)")

m_pc32 = np.maximum(np.real(m_f * np.conj(p32)), 0)
imshow_gray(m_pc32, "3.2: Phase-Compensated Image")

err_32 = np.abs(m_pc32 - m_ref_mri)
imshow_gray(err_32, "3.2: Error Image", vrange=[0, 0.2 * max_val])

print(f"PSNR: {psnr(m_ref_mri / max_val, m_pc32 / max_val, data_range=1.0):.2f} dB")
print(f"SSIM: {ssim(m_ref_mri / max_val, m_pc32 / max_val, data_range=1.0):.4f}")

# Save figures
plt.figure(); plt.imshow(np.angle(m_low32), cmap='gray'); plt.title('Phase Image (central 12.5%)'); plt.colorbar(); plt.axis('off')
plt.savefig('figures/p32_phase.png', bbox_inches='tight', dpi=150); plt.close()

plt.figure(); plt.imshow(m_pc32, cmap='gray'); plt.title('Phase-Compensated Image (12.5%)'); plt.axis('off')
plt.savefig('figures/p32_pc.png', bbox_inches='tight', dpi=150); plt.close()

plt.figure(); plt.imshow(err_32, cmap='gray', vmin=0, vmax=0.2 * max_val); plt.title('Error Image (12.5%)'); plt.axis('off')
plt.savefig('figures/p32_error.png', bbox_inches='tight', dpi=150); plt.close()

In [ ]:
# === PART 3.3: Phase-compensated image (central ±1/32, 6.25%, 16 PE lines) ===

half_w = Npe // 32  # 8
kdata_low33 = np.zeros_like(kdata)
kdata_low33[:, center - half_w : center + half_w] = kdata[:, center - half_w : center + half_w]

m_low33 = ifft2c(kdata_low33)
p33 = np.exp(1j * np.angle(m_low33))
imshow_gray(np.angle(m_low33), "3.3: Phase Image (central 6.25%)")

m_pc33 = np.maximum(np.real(m_f * np.conj(p33)), 0)
imshow_gray(m_pc33, "3.3: Phase-Compensated Image")

err_33 = np.abs(m_pc33 - m_ref_mri)
imshow_gray(err_33, "3.3: Error Image", vrange=[0, 0.2 * max_val])

print(f"PSNR: {psnr(m_ref_mri / max_val, m_pc33 / max_val, data_range=1.0):.2f} dB")
print(f"SSIM: {ssim(m_ref_mri / max_val, m_pc33 / max_val, data_range=1.0):.4f}")

# Save figures
plt.figure(); plt.imshow(np.angle(m_low33), cmap='gray'); plt.title('Phase Image (central 6.25%)'); plt.colorbar(); plt.axis('off')
plt.savefig('figures/p33_phase.png', bbox_inches='tight', dpi=150); plt.close()

plt.figure(); plt.imshow(m_pc33, cmap='gray'); plt.title('Phase-Compensated Image (6.25%)'); plt.axis('off')
plt.savefig('figures/p33_pc.png', bbox_inches='tight', dpi=150); plt.close()

plt.figure(); plt.imshow(err_33, cmap='gray', vmin=0, vmax=0.2 * max_val); plt.title('Error Image (6.25%)'); plt.axis('off')
plt.savefig('figures/p33_error.png', bbox_inches='tight', dpi=150); plt.close()

In [ ]:
# === PART 3.4: Phase-compensated image (central ±1/64, 3.125%, 8 PE lines) ===

half_w = Npe // 64  # 4
kdata_low34 = np.zeros_like(kdata)
kdata_low34[:, center - half_w : center + half_w] = kdata[:, center - half_w : center + half_w]

m_low34 = ifft2c(kdata_low34)
p34 = np.exp(1j * np.angle(m_low34))
imshow_gray(np.angle(m_low34), "3.4: Phase Image (central 3.125%)")

m_pc34 = np.maximum(np.real(m_f * np.conj(p34)), 0)
imshow_gray(m_pc34, "3.4: Phase-Compensated Image")

err_34 = np.abs(m_pc34 - m_ref_mri)
imshow_gray(err_34, "3.4: Error Image", vrange=[0, 0.2 * max_val])

print(f"PSNR: {psnr(m_ref_mri / max_val, m_pc34 / max_val, data_range=1.0):.2f} dB")
print(f"SSIM: {ssim(m_ref_mri / max_val, m_pc34 / max_val, data_range=1.0):.4f}")

# Save figures
plt.figure(); plt.imshow(np.angle(m_low34), cmap='gray'); plt.title('Phase Image (central 3.125%)'); plt.colorbar(); plt.axis('off')
plt.savefig('figures/p34_phase.png', bbox_inches='tight', dpi=150); plt.close()

plt.figure(); plt.imshow(m_pc34, cmap='gray'); plt.title('Phase-Compensated Image (3.125%)'); plt.axis('off')
plt.savefig('figures/p34_pc.png', bbox_inches='tight', dpi=150); plt.close()

plt.figure(); plt.imshow(err_34, cmap='gray', vmin=0, vmax=0.2 * max_val); plt.title('Error Image (3.125%)'); plt.axis('off')
plt.savefig('figures/p34_error.png', bbox_inches='tight', dpi=150); plt.close()

### Part 3.5–3.6: Comments on Phase-Compensated Image Quality

**Part 3.5 — Quality vs. central k-space ratio:**
As the central k-space ratio decreases (from 25% → 12.5% → 6.25% → 3.125%), the low-resolution phase estimate becomes less accurate. With 25% of k-space, the phase is smooth and well-estimated, yielding a phase-compensated image very close to the magnitude reference. As the ratio shrinks, the phase estimate captures less spatial variation, introducing residual phase errors that appear as signal loss or artifacts in the compensated image. Both PSNR and SSIM decrease with smaller ratios.

**Part 3.6 — Do PSNR and SSIM reflect visual observations?**
Yes. The visual degradation (increasing error, especially near edges and regions of rapid phase variation) is accurately captured by decreasing PSNR and SSIM values. The metrics track the trend well: larger central k-space ratios produce better phase estimates and therefore better compensated images, which is reflected in higher metric values.

In [ ]:
# === PART 4.1: Zero-Fill (ZF) Reconstruction — 5/8 Partial k-space ===

# Retain the first 5/8 of PE lines, zero the rest
Nacq = 5 * Npe // 8  # 160 acquired lines
kdata_partial = np.zeros_like(kdata)
kdata_partial[:, :Nacq] = kdata[:, :Nacq]

# Display k-space spectrum
imshow_kspace(kdata_partial, "4.1: k-space (5/8 partial, zero-filled)")

# ZF reconstruction (trivial: just IFFT of zero-filled k-space)
m_zf = np.abs(ifft2c(kdata_partial))
imshow_gray(m_zf, "4.1: ZF Reconstruction (magnitude)")

# Error image
err_41 = np.abs(m_zf - m_ref_mri)
imshow_gray(err_41, "4.1: Error Image", vrange=[0, 0.2 * max_val])

# PSNR and SSIM
psnr_41 = psnr(m_ref_mri / max_val, m_zf / max_val, data_range=1.0)
ssim_41 = ssim(m_ref_mri / max_val, m_zf / max_val, data_range=1.0)
print(f"PSNR: {psnr_41:.2f} dB")
print(f"SSIM: {ssim_41:.4f}")

# Save figures
plt.figure(); plt.imshow(np.log(np.abs(kdata_partial) + 1), cmap='gray'); plt.title('k-space (5/8 partial)'); plt.axis('off')
plt.savefig('figures/p41_kspace.png', bbox_inches='tight', dpi=150); plt.close()

plt.figure(); plt.imshow(m_zf, cmap='gray'); plt.title('ZF Reconstruction'); plt.axis('off')
plt.savefig('figures/p41_zf.png', bbox_inches='tight', dpi=150); plt.close()

plt.figure(); plt.imshow(err_41, cmap='gray', vmin=0, vmax=0.2 * max_val); plt.title('Error Image (ZF)'); plt.axis('off')
plt.savefig('figures/p41_error.png', bbox_inches='tight', dpi=150); plt.close()

In [ ]:
# === PART 4.2: POCS Reconstruction — 5/8 Partial k-space ===

# Use the phase from Part 3.1 (central ±1/8, best phase estimate)
p = p31  # unit-magnitude phase map from Part 3.1

# Acquired lines mask
acquired_mask = np.zeros((Nro, Npe), dtype=bool)
acquired_mask[:, :Nacq] = True

# Initialize: D_0 = zero-filled partial k-space
D_i = kdata_partial.copy()

n_iter = 5
psnr_pocs = []
ssim_pocs = []

for i in range(1, n_iter + 1):
    # Step 1: Image domain
    m_i = ifft2c(D_i)

    # Step 2: Phase-compensated output: |Re{m_i * p*}|
    m_out = np.abs(np.real(m_i * np.conj(p)))

    # Compute IQA for this iteration
    p_val = psnr(m_ref_mri / max_val, m_out / max_val, data_range=1.0)
    s_val = ssim(m_ref_mri / max_val, m_out / max_val, data_range=1.0)
    psnr_pocs.append(p_val)
    ssim_pocs.append(s_val)
    print(f"Iteration {i}: PSNR = {p_val:.2f} dB, SSIM = {s_val:.4f}")

    # Display output image
    imshow_gray(m_out, f"4.2 Iter {i}: Output Image")

    # Display k-space magnitude spectrum of output image
    D_out = fft2c(m_out)
    imshow_kspace(D_out, f"4.2 Iter {i}: k-space of Output Image")

    # Display error image
    err_i = np.abs(m_out - m_ref_mri)
    imshow_gray(err_i, f"4.2 Iter {i}: Error Image", vrange=[0, 0.2 * max_val])

    # Save figures for this iteration
    plt.figure(); plt.imshow(m_out, cmap='gray'); plt.title(f'POCS Iter {i}: Output'); plt.axis('off')
    plt.savefig(f'figures/p42_iter{i}_image.png', bbox_inches='tight', dpi=150); plt.close()

    plt.figure(); plt.imshow(np.log(np.abs(D_out) + 1), cmap='gray'); plt.title(f'POCS Iter {i}: k-space'); plt.axis('off')
    plt.savefig(f'figures/p42_iter{i}_kspace.png', bbox_inches='tight', dpi=150); plt.close()

    plt.figure(); plt.imshow(err_i, cmap='gray', vmin=0, vmax=0.2 * max_val); plt.title(f'POCS Iter {i}: Error'); plt.axis('off')
    plt.savefig(f'figures/p42_iter{i}_error.png', bbox_inches='tight', dpi=150); plt.close()

    # Step 3: Re-apply phase
    m_hat = m_out * p

    # Step 4: Back to k-space
    D_hat = fft2c(m_hat)

    # Step 5: Data consistency — replace acquired lines with original data
    D_i = D_hat.copy()
    D_i[acquired_mask] = kdata[acquired_mask]

# Print summary
print("\n--- POCS Summary ---")
for i in range(n_iter):
    print(f"Iter {i+1}: PSNR = {psnr_pocs[i]:.2f} dB, SSIM = {ssim_pocs[i]:.4f}")

In [ ]:
# === PART 4.3: 9/16 Partial k-space — ZF vs POCS (5th iteration) ===

ratio_label = "9/16"
Nacq_43 = 9 * Npe // 16  # 144 acquired lines
print(f"Nacq = {Nacq_43} (ratio = {ratio_label})")

# Create partial k-space
kdata_43 = np.zeros_like(kdata)
kdata_43[:, :Nacq_43] = kdata[:, :Nacq_43]

# --- ZF reconstruction ---
m_zf_43 = np.abs(ifft2c(kdata_43))
err_zf_43 = np.abs(m_zf_43 - m_ref_mri)
psnr_zf_43 = psnr(m_ref_mri / max_val, m_zf_43 / max_val, data_range=1.0)
ssim_zf_43 = ssim(m_ref_mri / max_val, m_zf_43 / max_val, data_range=1.0)
print(f"ZF  -> PSNR: {psnr_zf_43:.2f} dB, SSIM: {ssim_zf_43:.4f}")

# --- POCS (5 iterations, report 5th only) ---
mask_43 = np.zeros((Nro, Npe), dtype=bool)
mask_43[:, :Nacq_43] = True

D_i_43 = kdata_43.copy()
p = p31
for it in range(1, 6):
    m_i_43 = ifft2c(D_i_43)
    m_out_43 = np.abs(np.real(m_i_43 * np.conj(p)))
    m_hat_43 = m_out_43 * p
    D_hat_43 = fft2c(m_hat_43)
    D_i_43 = D_hat_43.copy()
    D_i_43[mask_43] = kdata[mask_43]

# POCS 5th iteration output
err_pocs_43 = np.abs(m_out_43 - m_ref_mri)
psnr_pocs_43 = psnr(m_ref_mri / max_val, m_out_43 / max_val, data_range=1.0)
ssim_pocs_43 = ssim(m_ref_mri / max_val, m_out_43 / max_val, data_range=1.0)
print(f"POCS -> PSNR: {psnr_pocs_43:.2f} dB, SSIM: {ssim_pocs_43:.4f}")

# --- Display ---
imshow_kspace(kdata_43, f"4.3: k-space ({ratio_label} partial)")
imshow_gray(m_zf_43, f"4.3: ZF Reconstruction")
imshow_gray(m_out_43, f"4.3: POCS (5th iter)")
imshow_gray(err_zf_43, f"4.3: Error (ZF)", vrange=[0, 0.2 * max_val])
imshow_gray(err_pocs_43, f"4.3: Error (POCS)", vrange=[0, 0.2 * max_val])

# k-space of POCS output
D_out_43 = fft2c(m_out_43)
imshow_kspace(D_out_43, f"4.3: k-space of POCS output")

# --- Save figures ---
plt.figure(); plt.imshow(np.log(np.abs(kdata_43)+1), cmap='gray'); plt.title(f'k-space ({ratio_label} partial)'); plt.axis('off')
plt.savefig('figures/p43_kspace.png', bbox_inches='tight', dpi=150); plt.close()

plt.figure(); plt.imshow(m_zf_43, cmap='gray'); plt.title('ZF Reconstruction'); plt.axis('off')
plt.savefig('figures/p43_zf.png', bbox_inches='tight', dpi=150); plt.close()

plt.figure(); plt.imshow(m_out_43, cmap='gray'); plt.title('POCS (5th iter)'); plt.axis('off')
plt.savefig('figures/p43_pocs.png', bbox_inches='tight', dpi=150); plt.close()

plt.figure(); plt.imshow(err_zf_43, cmap='gray', vmin=0, vmax=0.2*max_val); plt.title('Error (ZF)'); plt.axis('off')
plt.savefig('figures/p43_error_zf.png', bbox_inches='tight', dpi=150); plt.close()

plt.figure(); plt.imshow(err_pocs_43, cmap='gray', vmin=0, vmax=0.2*max_val); plt.title('Error (POCS)'); plt.axis('off')
plt.savefig('figures/p43_error_pocs.png', bbox_inches='tight', dpi=150); plt.close()

plt.figure(); plt.imshow(np.log(np.abs(D_out_43)+1), cmap='gray'); plt.title('k-space of POCS output'); plt.axis('off')
plt.savefig('figures/p43_kspace_pocs.png', bbox_inches='tight', dpi=150); plt.close()

In [ ]:
# === PART 4.4: 17/32 Partial k-space — ZF vs POCS (5th iteration) ===
ratio_label = "17/32"
Nacq_44 = 17 * Npe // 32  # 136 acquired lines
print(f"Nacq = {Nacq_44} (ratio = {ratio_label})")
kdata_44 = np.zeros_like(kdata)
kdata_44[:, :Nacq_44] = kdata[:, :Nacq_44]
m_zf_44 = np.abs(ifft2c(kdata_44))
err_zf_44 = np.abs(m_zf_44 - m_ref_mri)
psnr_zf_44 = psnr(m_ref_mri / max_val, m_zf_44 / max_val, data_range=1.0)
ssim_zf_44 = ssim(m_ref_mri / max_val, m_zf_44 / max_val, data_range=1.0)
print(f"ZF   -> PSNR: {psnr_zf_44:.2f} dB, SSIM: {ssim_zf_44:.4f}")
mask_44 = np.zeros((Nro, Npe), dtype=bool)
mask_44[:, :Nacq_44] = True
D_i_44 = kdata_44.copy()
p = p31
for it in range(1, 6):
    m_i_44 = ifft2c(D_i_44)
    m_out_44 = np.abs(np.real(m_i_44 * np.conj(p)))
    m_hat_44 = m_out_44 * p
    D_hat_44 = fft2c(m_hat_44)
    D_i_44 = D_hat_44.copy()
    D_i_44[mask_44] = kdata[mask_44]
err_pocs_44 = np.abs(m_out_44 - m_ref_mri)
psnr_pocs_44 = psnr(m_ref_mri / max_val, m_out_44 / max_val, data_range=1.0)
ssim_pocs_44 = ssim(m_ref_mri / max_val, m_out_44 / max_val, data_range=1.0)
print(f"POCS -> PSNR: {psnr_pocs_44:.2f} dB, SSIM: {ssim_pocs_44:.4f}")
imshow_kspace(kdata_44, f"4.4: k-space ({ratio_label} partial)")
imshow_gray(m_zf_44, f"4.4: ZF Reconstruction")
imshow_gray(m_out_44, f"4.4: POCS (5th iter)")
imshow_gray(err_zf_44, f"4.4: Error (ZF)", vrange=[0, 0.2 * max_val])
imshow_gray(err_pocs_44, f"4.4: Error (POCS)", vrange=[0, 0.2 * max_val])
D_out_44 = fft2c(m_out_44)
imshow_kspace(D_out_44, f"4.4: k-space of POCS output")
plt.figure(); plt.imshow(np.log(np.abs(kdata_44)+1), cmap='gray'); plt.title(f'k-space ({ratio_label} partial)'); plt.axis('off')
plt.savefig('figures/p44_kspace.png', bbox_inches='tight', dpi=150); plt.close()
plt.figure(); plt.imshow(m_zf_44, cmap='gray'); plt.title('ZF Reconstruction'); plt.axis('off')
plt.savefig('figures/p44_zf.png', bbox_inches='tight', dpi=150); plt.close()
plt.figure(); plt.imshow(m_out_44, cmap='gray'); plt.title('POCS (5th iter)'); plt.axis('off')
plt.savefig('figures/p44_pocs.png', bbox_inches='tight', dpi=150); plt.close()
plt.figure(); plt.imshow(err_zf_44, cmap='gray', vmin=0, vmax=0.2*max_val); plt.title('Error (ZF)'); plt.axis('off')
plt.savefig('figures/p44_error_zf.png', bbox_inches='tight', dpi=150); plt.close()
plt.figure(); plt.imshow(err_pocs_44, cmap='gray', vmin=0, vmax=0.2*max_val); plt.title('Error (POCS)'); plt.axis('off')
plt.savefig('figures/p44_error_pocs.png', bbox_inches='tight', dpi=150); plt.close()
plt.figure(); plt.imshow(np.log(np.abs(D_out_44)+1), cmap='gray'); plt.title('k-space of POCS output'); plt.axis('off')
plt.savefig('figures/p44_kspace_pocs.png', bbox_inches='tight', dpi=150); plt.close()

In [ ]:
# === PART 4.5: 33/64 Partial k-space — ZF vs POCS (5th iteration) ===
ratio_label = "33/64"
Nacq_45 = 33 * Npe // 64  # 132 acquired lines
print(f"Nacq = {Nacq_45} (ratio = {ratio_label})")
kdata_45 = np.zeros_like(kdata)
kdata_45[:, :Nacq_45] = kdata[:, :Nacq_45]
m_zf_45 = np.abs(ifft2c(kdata_45))
err_zf_45 = np.abs(m_zf_45 - m_ref_mri)
psnr_zf_45 = psnr(m_ref_mri / max_val, m_zf_45 / max_val, data_range=1.0)
ssim_zf_45 = ssim(m_ref_mri / max_val, m_zf_45 / max_val, data_range=1.0)
print(f"ZF   -> PSNR: {psnr_zf_45:.2f} dB, SSIM: {ssim_zf_45:.4f}")
mask_45 = np.zeros((Nro, Npe), dtype=bool)
mask_45[:, :Nacq_45] = True
D_i_45 = kdata_45.copy()
p = p31
for it in range(1, 6):
    m_i_45 = ifft2c(D_i_45)
    m_out_45 = np.abs(np.real(m_i_45 * np.conj(p)))
    m_hat_45 = m_out_45 * p
    D_hat_45 = fft2c(m_hat_45)
    D_i_45 = D_hat_45.copy()
    D_i_45[mask_45] = kdata[mask_45]
err_pocs_45 = np.abs(m_out_45 - m_ref_mri)
psnr_pocs_45 = psnr(m_ref_mri / max_val, m_out_45 / max_val, data_range=1.0)
ssim_pocs_45 = ssim(m_ref_mri / max_val, m_out_45 / max_val, data_range=1.0)
print(f"POCS -> PSNR: {psnr_pocs_45:.2f} dB, SSIM: {ssim_pocs_45:.4f}")
imshow_kspace(kdata_45, f"4.5: k-space ({ratio_label} partial)")
imshow_gray(m_zf_45, f"4.5: ZF Reconstruction")
imshow_gray(m_out_45, f"4.5: POCS (5th iter)")
imshow_gray(err_zf_45, f"4.5: Error (ZF)", vrange=[0, 0.2 * max_val])
imshow_gray(err_pocs_45, f"4.5: Error (POCS)", vrange=[0, 0.2 * max_val])
D_out_45 = fft2c(m_out_45)
imshow_kspace(D_out_45, f"4.5: k-space of POCS output")
plt.figure(); plt.imshow(np.log(np.abs(kdata_45)+1), cmap='gray'); plt.title(f'k-space ({ratio_label} partial)'); plt.axis('off')
plt.savefig('figures/p45_kspace.png', bbox_inches='tight', dpi=150); plt.close()
plt.figure(); plt.imshow(m_zf_45, cmap='gray'); plt.title('ZF Reconstruction'); plt.axis('off')
plt.savefig('figures/p45_zf.png', bbox_inches='tight', dpi=150); plt.close()
plt.figure(); plt.imshow(m_out_45, cmap='gray'); plt.title('POCS (5th iter)'); plt.axis('off')
plt.savefig('figures/p45_pocs.png', bbox_inches='tight', dpi=150); plt.close()
plt.figure(); plt.imshow(err_zf_45, cmap='gray', vmin=0, vmax=0.2*max_val); plt.title('Error (ZF)'); plt.axis('off')
plt.savefig('figures/p45_error_zf.png', bbox_inches='tight', dpi=150); plt.close()
plt.figure(); plt.imshow(err_pocs_45, cmap='gray', vmin=0, vmax=0.2*max_val); plt.title('Error (POCS)'); plt.axis('off')
plt.savefig('figures/p45_error_pocs.png', bbox_inches='tight', dpi=150); plt.close()
plt.figure(); plt.imshow(np.log(np.abs(D_out_45)+1), cmap='gray'); plt.title('k-space of POCS output'); plt.axis('off')
plt.savefig('figures/p45_kspace_pocs.png', bbox_inches='tight', dpi=150); plt.close()

# === SUMMARY TABLE ===
print("\n" + "="*70)
print("SUMMARY: ZF vs POCS (5th iter) across partial k-space ratios")
print("="*70)
print(f"{'Part':<8} {'Ratio':<8} {'Nacq':<6} {'ZF PSNR':>10} {'ZF SSIM':>10} {'POCS PSNR':>11} {'POCS SSIM':>11}")
print("-"*70)
print(f"{'4.1/4.2':<8} {'5/8':<8} {160:<6} {psnr_41:>10.2f} {ssim_41:>10.4f} {psnr_pocs[-1]:>11.2f} {ssim_pocs[-1]:>11.4f}")
print(f"{'4.3':<8} {'9/16':<8} {Nacq_43:<6} {psnr_zf_43:>10.2f} {ssim_zf_43:>10.4f} {psnr_pocs_43:>11.2f} {ssim_pocs_43:>11.4f}")
print(f"{'4.4':<8} {'17/32':<8} {Nacq_44:<6} {psnr_zf_44:>10.2f} {ssim_zf_44:>10.4f} {psnr_pocs_44:>11.2f} {ssim_pocs_44:>11.4f}")
print(f"{'4.5':<8} {'33/64':<8} {Nacq_45:<6} {psnr_zf_45:>10.2f} {ssim_zf_45:>10.4f} {psnr_pocs_45:>11.2f} {ssim_pocs_45:>11.4f}")